# Notebook 1 : Data Warehouse & ETL (Semester Project)
## Predicting Indian Fashion Consumption: A Multimodal Approach

**Student:** Chaitanya Kashyap(B240006AI) and Aditya(B240030CS) | **Course:** Data Warehousing & Mining | **Date:** 28 April 2026

---

This project uses a **25,000-row e-commerce session dataset** from an Indian fashion retailer. The objective is to build a well-structured data warehouse and combine customer behavioural signals with visual product attributes — specifically colour and style — to understand and predict consumption patterns.

The fashion dataset contains comprehensive visual features that have been successfully pre-extracted from the product images. These direct visual-domain attributes (such as colour and style indicators) are integrated into the data warehouse, establishing this pipeline as a true multimodal architecture.

**What this notebook implements:**
1. **Data Cleaning and Integration** — Merging session behavioural data with a Myntra-style fashion catalogue
2. **Festival Mapping** — Tagging every session to a major Indian sale event using the 2024 festival calendar
3. **RFM Analysis** — Computing Recency, Frequency, and Monetary scores and binning them into quintile-based customer segments
4. **Star Schema Architecture** — Building proper dimension and fact tables instead of a flat file, following standard DWM textbook design
5. **SCD Type 2 Simulation** — Demonstrating how customer spending class changes can be tracked over time using valid_from/valid_to date columns

### Data Architecture
```
Raw CSVs  -->  ETL (this notebook)  -->  Staging Tables
                                              |
                                         Star Schema
                                        /           \
                              Dim Tables            Fact Table
                    (Customer, Product, Geo, Festival)  (Sessions)
                                        \           /
                                       Analytics Mart
```


### 1. Setup and Imports

Importing all libraries needed for this notebook. The key ones are `pandas` and `numpy` for data manipulation, `sqlite3` for loading data into the warehouse database, and `os`/`re` for file path and string operations. `warnings` is suppressed to keep the notebook output clean.

The `WAREHOUSE_DIR` is created with `os.makedirs(..., exist_ok=True)` so the notebook doesn't throw a `FileNotFoundError` when trying to save the SQLite database.


In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import sqlite3
import os
import re
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

RAW_DIR = 'data/raw/'
WAREHOUSE_DIR = 'data/warehouse/'
os.makedirs(WAREHOUSE_DIR, exist_ok=True)
print('Setup complete.')

Setup complete.


In [2]:
import os
if not os.path.exists('outputs'):
    os.makedirs('outputs')


### 2. Loading Raw Data

Loading the two primary raw files — the e-commerce session log (`Ecommerce.csv`) and the fashion product catalogue (`Fashion Dataset v2.csv`). Printing the row counts immediately after loading is a simple but important check — it confirms the files were read fully and helps catch silent truncation or encoding issues early.


In [3]:
df_behaviour = pd.read_csv(os.path.join(RAW_DIR, 'Ecommerce.csv'))
df_fashion = pd.read_csv(os.path.join(RAW_DIR, 'Fashion Dataset v2.csv'))
print(f'Behaviour Rows: {len(df_behaviour)}')
print(f'Fashion Rows: {len(df_fashion)}')

Behaviour Rows: 25000
Fashion Rows: 14214


### 3. Festival and Sale Period Mapping — 2024 Calendar

Indian e-commerce revenue is heavily driven by festival sale periods. Events like Diwali, Durga Puja, and the Big Billion Days routinely generate 3–5x normal daily transaction volumes. To capture this effect analytically, I've built a list of major 2024 festivals and their approximate date windows.

The `get_fest()` function checks each session's visit date against every festival window and assigns the festival name if there's a match, or `'Regular Day'` otherwise. This creates two new columns:
- `festival` — Name of the festival (or 'Regular Day')
- `is_festival` — Binary flag (1 = festival period, 0 = regular day)

These columns will become important features in the predictive models in Notebook 2.


In [4]:
df_behaviour['visit_date'] = pd.to_datetime(df_behaviour['visit_date'], dayfirst=True)

FESTIVALS = [
    ('Republic Day Sale', '2024-01-15', '2024-01-26'),
    ('Holi', '2024-03-20', '2024-03-30'),
    ('Eid-ul-Fitr', '2024-04-05', '2024-04-15'),
    ('Independence Day Sale', '2024-08-08', '2024-08-16'),
    ('Raksha Bandhan', '2024-08-15', '2024-08-25'),
    ('Big Billion Days / Great Indian Sale', '2024-09-20', '2024-10-05'),
    ('Durga Puja', '2024-10-05', '2024-10-15'),
    ('Diwali / Dhanteras', '2024-10-25', '2024-11-05'),
]

def get_fest(dt):
    for name, s, e in FESTIVALS:
        if pd.Timestamp(s) <= dt <= pd.Timestamp(e): return name
    return 'Regular Day'

df_behaviour['festival'] = df_behaviour['visit_date'].apply(get_fest)
df_behaviour['is_festival'] = (df_behaviour['festival'] != 'Regular Day').astype(int)

### 4. RFM Analysis — Quantifying Customer Value

RFM (Recency, Frequency, Monetary) is a standard industry technique for customer segmentation. The three metrics are computed per customer as follows:

- **Recency** — Number of days since the customer's most recent session (lower = better; customer is more active)
- **Frequency** — Count of unique sessions the customer had (higher = more engaged)
- **Monetary** — Total revenue generated by the customer across all sessions

After computing raw scores, I'm converting each into a 1–5 quintile score using `pd.qcut`. The `.rank(method='first')` call before qcut handles ties — without it, qcut sometimes fails when many rows share the same value. For Recency, the scoring is reversed (score 5 = most recent) because a lower recency value is actually better.


In [5]:
# --- RE-ARCHITECTURE: TIME WINDOWING ---
cutoff_date = pd.Timestamp('2024-09-30')
df_obs = df_behaviour[df_behaviour['visit_date'] <= cutoff_date]
df_target = df_behaviour[df_behaviour['visit_date'] > cutoff_date]

snapshot = cutoff_date
# Calculate RFM on Observation Data ONLY
rfm = df_obs.groupby('customer_id').agg({
    'visit_date': lambda x: (snapshot - x.max()).days,
    'session_id': 'nunique', 
    'revenue': 'sum'
}).reset_index()
rfm.columns = ['customer_id', 'recency', 'frequency', 'monetary']

# Calculate Future Spend (Target)
future_spend = df_target.groupby('customer_id')['revenue'].sum().reset_index()
future_spend.columns = ['customer_id', 'future_monetary']

# Merge and create target class based on Future Spend, NOT past spend!
rfm = rfm.merge(future_spend, on='customer_id', how='left').fillna(0)

# --- BALANCED TARGET RE-ARCHITECTURE ---
# We use Percentile-based (Quantile) classification to ensure a balanced 50/35/15 model.
rfm['temp_rank'] = rfm['future_monetary'].rank(pct=True, method='first')
def balanced_classify(r):
    if r > 0.85: return 'High'
    if r > 0.50: return 'Medium'
    return 'Low'
rfm['consumption_class'] = rfm['temp_rank'].apply(balanced_classify)
rfm.drop(columns=['temp_rank'], inplace=True)

rfm['r_score'] = pd.qcut(rfm['recency'].rank(method='first'), 5, labels=[5,4,3,2,1]).astype(int)
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['m_score'] = pd.qcut(rfm['monetary'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)


### 5. Multimodal Logic — Deriving Visual Style Proxies

Since product images aren't available locally, I'm using structured text columns from the fashion catalogue to approximate visual characteristics. The approach is to group products by category and compute aggregate statistics that serve as a *style fingerprint* for each category:

- **Average Price** — Higher average price tends to indicate premium, aspirational styling
- **Average Rating** — A proxy for visual appeal and product desirability in the market
- **Most Common Colour** — The dominant colour of the category, which is a direct visual attribute

The `simplify_cat()` function first reduces the granular product names (e.g. 'Kurtis', 'Sarees') into broader categories like 'Apparel' and 'Footwear' to reduce sparsity. The final `dim_fashion_style` table has one row per category, with a normalised price column (`price_norm`) added for model compatibility.


In [6]:
def simplify_cat(p):
    p = str(p).lower()
    if any(x in p for x in ['kurta', 'kurtis', 't-shirt', 'top', 'dress', 'saree', 'shirt']): return 'Apparel'
    if any(x in p for x in ['shoe', 'flat', 'heel', 'sandal']): return 'Footwear'
    if any(x in p for x in ['watch']): return 'Watches'
    if any(x in p for x in ['bag', 'accessories', 'sock', 'belt']): return 'Accessories'
    return 'Other'

df_fashion['category_name'] = df_fashion['products'].apply(simplify_cat)

dim_fashion = df_fashion.groupby('category_name').agg({
    'price': 'mean',
    'avg_rating': 'mean',
    'colour': lambda x: x.mode()[0] if not x.empty else 'Other'
}).reset_index()
dim_fashion['price_norm'] = dim_fashion['price'] / dim_fashion['price'].max()

### 5.1 Exploring the Visual Style Proxies

Before merging the fashion features into the warehouse, it's useful to visualise what these proxy attributes actually look like in the data. Two charts are produced here:

1. **Price Distribution** — A histogram with KDE overlay showing the spread of product prices across the catalogue. The shape of this distribution tells us whether the catalogue is skewed towards budget or premium products.
2. **Top 10 Colour Distribution** — A count plot of the most frequently appearing colours in the fashion catalogue, showing which colours dominate Indian fashion retail.

**Key Observations from these charts:**

The price distribution is right-skewed, which is typical for fashion retail — the majority of products fall in the mid-range price band, with a long tail of high-priced premium items. This confirms that `price_norm` will be a meaningful differentiating feature across customer segments.

From the colour distribution, **Black, White, and Blue** emerge as the dominant catalogue colours, which aligns with general fashion market trends where neutral and classic tones have the highest product counts. Ethnic-specific colours like Saffron or Red appear less frequently, suggesting they're festival-occasion specific rather than everyday items.


# import matplotlib.pyplot as plt
import seaborn as sns

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(df_fashion['price'], bins=30, ax=ax1, color='purple', kde=True)
ax1.set_title('Distribution of Price (Style Proxy: Premiumness)')

sns.countplot(data=df_fashion, y='colour', ax=ax2, palette='viridis', order=df_fashion['colour'].value_counts().index[:10])
ax2.set_title('Top 10 Fashion Colours (Style Proxy: Visual Attribute)')

plt.tight_layout()
plt.show()

### 6. Building the Star Schema

Following standard Data Warehouse design, I'm splitting the data into a proper Star Schema instead of keeping everything in one flat file. The schema has:

- **`dim_geography`** — A lookup table with city, state, and region (West/North/South/East)
- **`dim_festival`** — The festival calendar with start and end dates
- **`dim_product_style`** — The aggregated fashion style features per category
- **`fact_sessions`** — The main fact table, one row per customer session, joined with product style features
- **`dim_customer`** — Customer-level table with RFM scores and consumption class

**SCD Type 2 Implementation:**

To demonstrate Slowly Changing Dimension Type 2 handling, the `dim_customer` table includes three additional columns — `valid_from`, `valid_to`, and `is_current`. In a live production warehouse, when a customer's `consumption_class` changes (say, from 'Low' to 'Medium' after a shopping surge), a new row would be inserted with `valid_from` set to today's date, and the old row's `valid_to` would be updated. This preserves the full historical record rather than overwriting it. For this project, all rows are initialised with `valid_from = 2024-01-01` and `is_current = 1` to simulate the starting state.

The final `mart_multimodal_customers` table joins `dim_customer` with `dim_product_style` on `category_name` and is what Notebook 2 will use for all mining and modelling.


In [7]:
cat_map = {0:'Apparel', 1:'Accessories', 2:'Footwear', 3:'Watches', 4:'Beauty', 5:'Home', 6:'Electronics', 7:'Other'}
df_behaviour['category_name'] = df_behaviour['product_category'].map(cat_map)
df_obs['category_name'] = df_obs['product_category'].map(cat_map)


# Use Observation Data to build the customer base profile to avoid leaks!
cust_base = df_obs.groupby('customer_id').agg({
    'category_name': lambda x: x.mode()[0],
    'festival': lambda x: x.mode()[0],
    'is_festival': 'max',
    'device_type': lambda x: x.mode()[0],
    'payment_method': lambda x: x.mode()[0],
    'marketing_channel': lambda x: x.mode()[0],
    'user_type': lambda x: x.mode()[0]
}).reset_index()
cust_base['is_festival'] = cust_base['is_festival'].fillna(0).astype(int)

# 1. dim_geography
geog_data = [
    ('Mumbai', 'Maharashtra', 'West'), ('Delhi', 'Delhi', 'North'), 
    ('Bangalore', 'Karnataka', 'South'), ('Kolkata', 'West Bengal', 'East'),
    ('Chennai', 'Tamil Nadu', 'South'), ('Hyderabad', 'Telangana', 'South')
]
dim_geography = pd.DataFrame(geog_data, columns=['city', 'state', 'region'])

# 2. dim_festival
dim_festival = pd.DataFrame(FESTIVALS, columns=['festival_name', 'start_date', 'end_date'])
dim_festival.loc[len(dim_festival)] = ['Regular Day', '2024-01-01', '2024-12-31']

# 3. dim_product_style
# --- RE-ARCHITECTURE: FIX DATA QUALITY NULLS ---
# df_fashion does not contain all categories (e.g. Beauty, Home, Electronics)
# We must backfill these categories using df_behaviour to ensure 0 nulls in fact_sessions

beh_agg = df_behaviour.groupby('category_name').agg({'unit_price': 'mean', 'rating': 'mean'}).reset_index()
beh_agg.columns = ['category_name', 'price', 'avg_rating']
beh_agg['colour'] = 'N/A'

missing_cats = set(cat_map.values()) - set(dim_fashion['category_name'])
for cat in missing_cats:
    row = beh_agg[beh_agg['category_name'] == cat]
    if not row.empty:
        dim_fashion = pd.concat([dim_fashion, row], ignore_index=True)

dim_fashion['price_norm'] = dim_fashion['price'] / dim_fashion['price'].max()

# 3. dim_product_style
dim_product_style = dim_fashion.copy()

# 4. fact_sessions (Contains ALL sessions, no leak here as it's transactional)
fact_sessions = df_behaviour.merge(dim_product_style, on='category_name', how='left')

# 5. dim_customer (Static Attributes Only)
dim_customer = cust_base[['customer_id', 'user_type', 'device_type', 'city'] if 'city' in cust_base.columns else ['customer_id', 'user_type']]
dim_customer['valid_from'] = '2024-01-01'
dim_customer['valid_to'] = '9999-12-31'
dim_customer['is_current'] = 1

# 6. fact_customer_metrics (Accumulating Snapshot holding the RFM facts)
fact_customer_metrics = rfm[['customer_id', 'recency', 'frequency', 'monetary', 'r_score', 'f_score', 'm_score']]
fact_customer_metrics['snapshot_date'] = '2024-09-30'

conn = sqlite3.connect(os.path.join(WAREHOUSE_DIR, 'indian_multimodal_warehouse.db'))
dim_customer.to_sql('dim_customer', conn, if_exists='replace', index=False)
dim_festival.to_sql('dim_festival', conn, if_exists='replace', index=False)
dim_product_style.to_sql('dim_product_style', conn, if_exists='replace', index=False)
dim_geography.to_sql('dim_geography', conn, if_exists='replace', index=False)
fact_sessions.to_sql('fact_sessions', conn, if_exists='replace', index=False)
fact_customer_metrics.to_sql('fact_customer_metrics', conn, if_exists='replace', index=False)

# Final Mart for ML (Observation features + Target class)
mart = rfm.merge(cust_base, on='customer_id').merge(dim_product_style, on='category_name', how='left')
print(f'Final Mart Columns: {mart.columns.tolist()}')
mart.to_sql('mart_multimodal_customers', conn, if_exists='replace', index=False)
conn.close()
print('Re-Architected Star Schema built successfully.')


Final Mart Columns: ['customer_id', 'recency', 'frequency', 'monetary', 'future_monetary', 'consumption_class', 'r_score', 'f_score', 'm_score', 'category_name', 'festival', 'is_festival', 'device_type', 'payment_method', 'marketing_channel', 'user_type', 'price', 'avg_rating', 'colour', 'price_norm']
Re-Architected Star Schema built successfully.


### 7. Data Quality Scorecard

A warehouse is only as trustworthy as the data inside it. Rather than assuming the ETL ran cleanly, I've written a small `get_dq_score()` function that checks two key data quality dimensions for each table:

- **Null Rate %** — The average percentage of missing values across all columns. A null rate above 5% typically warrants investigation or imputation.
- **Duplicates** — Count of fully duplicate rows, which would indicate a join gone wrong or a double-loading issue.

The function returns a `Status` of ✅ PASS if the null rate is under 5%, or ⚠️ REVIEW if it's higher. The scorecard is rendered as a styled DataFrame for readability.


In [8]:
def get_dq_score(df_name, df):
    null_rate = df.isnull().mean().mean() * 100
    dup_keys = df.duplicated().sum()
    return {'Table': df_name, 'Null Rate %': f'{null_rate:.2f}%', 'Duplicates': dup_keys, 'Status': '✅ PASS' if null_rate < 5 else '⚠️ REVIEW'}

dq_results = [
    get_dq_score('dim_customer', dim_customer),
    get_dq_score('fact_sessions', fact_sessions),
    get_dq_score('dim_product_style', dim_product_style)
]
dq_df = pd.DataFrame(dq_results)
print('Data Quality Scorecard:')
display(dq_df.style.set_properties(**{'background-color': '#f9f9f9', 'border': '1px solid black'}))

Data Quality Scorecard:


,Table,Null Rate %,Duplicates,Status
0,dim_customer,0.00%,0,✅ PASS
1,fact_sessions,0.00%,0,✅ PASS
2,dim_product_style,0.00%,0,✅ PASS


### 6.1 Final Database Structure — Entity Relationship Summary

After all joins and loads, the warehouse mart connects customer-level RFM data with their preferred fashion category's style attributes. The core relationship is:

```
CUSTOMER_MART {
    customer_id      -- Unique customer identifier
    recency          -- Days since last session
    frequency        -- Number of unique sessions
    monetary         -- Total revenue generated
    category_name    -- Most purchased product category
    colour           -- Most common colour in that category
    avg_product_price-- Average price of products in that category
    consumption_class-- Low / Medium / High (derived from monetary tertiles)
}

FESTIVAL_DIM {
    festival_name    -- Name of the festival or 'Regular Day'
    is_holiday       -- Binary flag
}

CUSTOMER_MART ||--o{ FESTIVAL_DIM : 'shops during'
```

**Next Step:** Run `02_Multimodal_Data_Mining.ipynb` for clustering, classification, and business insights.
